# Alineación descriptiva con movimiento absoluto al vencimiento

## Objetivo

Alinear cada straddle previamente seleccionado con el cierre SPY **sin ajustar** observado exactamente en su vencimiento. Para un par disponible se define `terminal_move_rel = abs(SPY_close_T - K) / SPY_close_t`. Representa el valor intrínseco conjunto de call y put al vencimiento, normalizado por el spot de t; no es P&L realizado de una operación.

## Reglas que no cambian

La selección de contratos, 27–33 DTE y filtros de la serie original no se recalculan. Se exige un cierre de SPY en la fecha T exacta: no se adelanta ni arrastra un precio. Las filas sin straddle continúan en la salida. Si T queda fuera del historial, se marca censura final; si T está dentro pero no hay cierre exacto, se conserva como ausencia explícita.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root / 'scripts'))
from align_straddle_terminal_move import align_terminal_move
print('Lógica reutilizable: scripts/align_straddle_terminal_move.py')

Lógica reutilizable: scripts/align_straddle_terminal_move.py


In [2]:
result = align_terminal_move(write_csv=True)
aligned = result['aligned'].copy()
aligned['observation_date'] = pd.to_datetime(aligned['observation_date'])
aligned['expiration'] = pd.to_datetime(aligned['expiration'])
print('Entrada congelada:', result['input_path'])
print('Cierres SPY:', result['price_path'])
print('CSV alineado:', result['output_path'])
display(pd.DataFrame([result['audit']]))

Entrada congelada: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\opciones_notas\straddle_ATM_SPY_volatilidad_futura\data\derived\spy_atm30_straddle_mid_rel_2020-01-02_2026-08-31.csv
Cierres SPY: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\HMM_regime_options\data\derived\spy_daily_adjusted_2020_2026.csv
CSV alineado: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\opciones_notas\straddle_ATM_SPY_volatilidad_futura\data\derived\spy_atm30_straddle_terminal_move_rel_2020-01-02_2026-08-31.csv


,rows_total,pairs_available,expiration_closes_observed,censored_at_history_end,no_exact_expiration_close,no_straddle_available,price_history_start,price_history_end
0,1674,1672,1652,20,0,2,2020-01-02,2026-08-31


### Comprobación de cobertura

Los vencimientos observados se cuentan aparte de pares disponibles. La censura final no se trata como cero ni se sustituye con la siguiente sesión.

In [3]:
reason_counts = (aligned['terminal_absence_reason']
    .fillna('observed_exact_expiration_close')
    .value_counts(dropna=False)
    .rename_axis('terminal_status')
    .reset_index(name='rows'))
display(reason_counts)

coverage_by_year = (aligned.assign(year=aligned['observation_date'].dt.year)
    .groupby('year', as_index=False)
    .agg(rows=('series_row_index', 'size'), pairs_available=('pair_available', 'sum'),
         expiration_closes_observed=('terminal_followup_complete', 'sum'),
         censored_at_history_end=('censored_at_history_end', 'sum')))
display(coverage_by_year)

,terminal_status,rows
0,observed_exact_expiration_close,1652
1,expiration_after_available_price_history,20
2,no_straddle_available,2


,year,rows,pairs_available,expiration_closes_observed,censored_at_history_end
0,2020,253,252,252,0
1,2021,252,251,251,0
2,2022,251,251,251,0
3,2023,250,250,250,0
4,2024,252,252,252,0
5,2025,250,250,250,0
6,2026,166,166,146,20


## Ejemplos de índices de sesión y fechas

Se muestran el primer par disponible, el par disponible central y el último par disponible. `series_row_index` es el índice cero-basado de la fila original de la serie; cada ejemplo deja visibles t, T, K y los dos cierres. El último caso permite comprobar explícitamente la regla de censura si su vencimiento supera el historial de precios.

In [4]:
pairs = aligned.loc[aligned['pair_available']].copy()
examples = pairs.iloc[[0, (len(pairs) - 1) // 2, -1]]
example_columns = [
    'series_row_index', 'observation_date', 'expiration', 'dte', 'strike',
    'spy_close_t', 'spy_close_T', 'terminal_move_rel',
    'terminal_followup_complete', 'terminal_absence_reason', 'censored_at_history_end',
    'call_contractID', 'put_contractID'
]
display(examples[example_columns])

complete_checks = aligned.loc[aligned['terminal_followup_complete'], ['observation_date', 'expiration', 'spy_close_t', 'strike', 'spy_close_T', 'terminal_move_rel']]
assert complete_checks['spy_close_T'].notna().all()
assert complete_checks['terminal_move_rel'].notna().all()
assert (aligned.loc[aligned['censored_at_history_end'], 'spy_close_T'].isna()).all()
print('Comprobaciones de cierre exacto y censura final: correctas.')

,series_row_index,observation_date,expiration,dte,strike,spy_close_t,spy_close_T,terminal_move_rel,terminal_followup_complete,terminal_absence_reason,censored_at_history_end,call_contractID,put_contractID
0,0,2020-01-02,2020-01-31,29.0,325.0,324.87,321.73,0.010066,True,None,False,SPY200131C00325000,SPY200131P00325000
837,837,2023-05-01,2023-06-02,32.0,416.0,415.51,427.92,0.028688,True,None,False,SPY230602C00416000,SPY230602P00416000
1673,1673,2026-08-31,2026-09-30,30.0,767.0,767.05,NaN,NaN,False,expiration_after_available_price_history,True,SPY260930C00767000,SPY260930P00767000


Comprobaciones de cierre exacto y censura final: correctas.


## Límites

Esta tabla no construye el benchmark histórico, no calcula correlaciones ni métricas de predicción, y no interpreta la diferencia entre prima y valor intrínseco como rentabilidad. SPY tiene opciones americanas y las cotizaciones archivadas no prueban ejecución simultánea. La hora EOD de las cadenas continúa sin verificarse.

## Mis notas

- 
- 
- 